<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 01
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Phân tích thị trường mỹ phẩm nội vs ngoại trên Tiki"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  TIỀN XỬ LÝ BỘ DỮ LIỆU TỪ TIKI
</div>

## **1. NHẬP DỮ LIỆU VÀ KIỂM TRA TỔNG QUAN**
### **1.1. Mục tiêu**
Bước đầu tiên trong quy trình tiền xử lý là khảo sát toàn diện tập dữ liệu thô trước khi thực hiện bất kỳ thao tác biến đổi nào. Mục tiêu của bước này là xác định cấu trúc, kiểm tra tính toàn vẹn và phát hiện các đặc điểm thống kê quan trọng. Đây là bước đệm để đưa ra các quyết định xử lý dữ liệu và kỹ thuật tính toán ở các bước tiếp theo.

### **1.2. Tổng quan cấu trúc và Kiểu dữ liệu**
Tập dữ liệu có kích thước **(8041, 21)**, trải rộng từ thông tin định danh, thương hiệu đến các chỉ số kinh doanh và tương tác khách hàng. 

Tập dữ liệu được chia thành 4 nhóm chính để phục vụ cho các mục tiêu phân tích khác nhau:

* **Nhóm định danh và phân loại:** Dùng để định danh duy nhất từng sản phẩm và phân loại các nhóm hàng trên thị trường.
    * *Thuộc tính:* `product_id`, `name`, `name_clean`, `brand_name`, `category`, `origin`, `product_type`.
* **Nhóm chỉ số Kinh doanh và giá cả:** Phản ánh giá trị kinh tế, chiến lược định giá và doanh thu dự kiến của sản phẩm.
    * *Thuộc tính:* `price`, `original_price`, `discount_rate`, `estimated_revenue`, `price_segment`.
* **Nhóm tương tác và độ phổ biến:** Đo lường sức hút, mức độ quan tâm và niềm tin của người tiêu dùng đối với sản phẩm.
    * *Thuộc tính:* `rating`, `review_count`, `sold_count`, `popularity_tier`, `rating_tier`.
* **Nhóm chỉ số niềm tin và trạng thái:** Xác định uy tín của nhà bán hàng và tình trạng sẵn có của hàng hóa trên sàn.
    * *Thuộc tính:* `is_official_store`, `is_authentic`, `tiki_verified`, `has_authentic_badge`, `availability_label`.

### **1.3. Phân tích giá trị thiếu**
Dữ liệu có độ sạch rất cao về mặt hiện diện. Qua kiểm tra, chỉ có duy nhất một thuộc tính gặp vấn đề về tính toàn vẹn:

| Thuộc tính | Tỉ lệ thiếu | Số lượng (ước tính) | Hướng xử lý |
| :--- | :--- | :--- | :--- |
| **`origin`** | **5,3%** | ~431 | Suy luận logic dựa trên `is_imported` và `brand_name`. |

### **1.4. Thống kê mô tả và nhận diện bất thường**
Phân tích các chỉ số tập trung và phân tán bộc lộ những đặc điểm đặc thù của ngành hàng mỹ phẩm:
* **Phân khúc giá**
    * Phân phối lệch phải mạnh: Giá trung bình (~597.727đ) cao gấp đôi mức giá trung vị (298.000đ). Điều này chứng tỏ phần lớn thị trường tập trung ở phân khúc bình dân, nhưng bị kéo lệch bởi một nhóm nhỏ sản phẩm cao cấp có giá trị lên tới 18.000.000đ.
    * Chiến lược giảm giá: Hơn 50% sản phẩm có `discount_rate` bằng 0.
* **Lượt bán**
    * Biến động cực đại: Đây là cột có độ phân tán lớn nhất (Std = 8.378).
    * Khoảng cách cực lớn: Trong khi 75% sản phẩm chỉ bán được dưới 10 lượt, thì giá trị lớn nhất đạt tới 548.710. Điều này khẳng định sự tồn tại của nhóm sản phẩm bán chạy đột biến. Nhóm này cần được xử lý ngoại lai (outliers) kỹ lưỡng để không làm sai lệch các mô hình phân tích sau này.
* **Tương tác khách hàng**
    * Sản phẩm mới chiếm ưu thế: Cả `rating` và `review_count` đều có trung vị bằng 0. 
    * Đặc trưng TMĐT: Điều này phản ánh thực tế là có rất nhiều sản phẩm mới được đăng bán hoặc các sản phẩm ngách ít người mua. Tuy nhiên, việc `rating` đạt tối đa 5.0 và `review_count` tối đa 1.651 cho thấy nhóm sản phẩm có thương hiệu uy tín có mức độ tương tác và hài lòng rất cao.



In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

path = "../data" 
df = pd.read_csv(f"{path}/tiki_cosmetics_raw.csv")

print(f"Shape: {df.shape}")
print(f"\n--- Kiểu dữ liệu")
print(df.dtypes)
print(f"\n--- Tỉ lệ missing (%)")
missing = (df.isnull().mean() * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False))
print(f"\n--- Thống kê cơ bản")
print(df[["price","original_price","discount_rate","rating","review_count","sold_count"]].describe())

Shape: (8041, 21)

--- Kiểu dữ liệu
product_id               int64
name                    object
brand_id               float64
brand_name              object
seller_id                int64
seller_name             object
category                object
primary_category        object
price                    int64
original_price           int64
discount_rate            int64
rating                 float64
review_count             int64
sold_count               int64
origin                  object
is_imported               bool
is_authentic             int64
is_official_store         bool
tiki_verified            int64
has_authentic_badge       bool
availability             int64
dtype: object

--- Tỉ lệ missing (%)
origin    5.0
dtype: float64

--- Thống kê cơ bản
              price  original_price  discount_rate       rating  review_count  \
count  8.041000e+03    8.041000e+03    8041.000000  8041.000000   8041.000000   
mean   6.091735e+05    6.552813e+05       7.680015     1.617311 

## **2. LẤP ĐẦY DỮ LIỆU THIẾU VÀ CHUẨN HÓA CƠ BẢN**

### **2.1. Xử lý giá trị thiếu và trùng lặp**
Chiến lược xử lý được xây dựng dựa trên việc bảo toàn tối đa dữ liệu thay vì loại bỏ các dòng có giá trị trống (Missing values).

#### **2.1.1. Khôi phục xuất xứ từ thuộc tính nhị phân**
Cột `origin` là trường duy nhất có dữ liệu thiếu (5,3%). Nhóm đã triển khai hàm `fill_origin` để suy luận giá trị dựa trên cột `is_imported`:
* Nếu `is_imported` là `True`: Điền **"Ngoài nước (không rõ)"**.
* Nếu `is_imported` là `False`: Điền **"Việt Nam"**.
* Các trường hợp còn lại giữ nhãn **"Không rõ"** để xử lý thủ công ở bước chuẩn hóa xuất xứ.

#### **2.1.2. Thiết lập giá trị mặc định cho chỉ số tương tác**
Đối với các cột định lượng phản ánh hành vi người dùng (`rating`, `review_count`, `sold_count`), nhóm quyết định điền giá trị 0 thay vì xóa bỏ. Quyết định này phản ánh đúng thực tế nghiệp vụ: các sản phẩm mới niêm yết thường chưa có lượt mua hoặc đánh giá.

#### **2.1.3. Đồng bộ hóa thương hiệu và tính toán lại tỉ lệ giảm giá**
* **Thương hiệu:** Các trường `brand_name` trống được điền tạm thời là **"Không rõ"** và chuẩn hóa các lỗi nhập liệu phổ biến như chuyển "Viet Nam" thành "Việt Nam".
* **Tỉ lệ giảm giá:** Nhóm đã tái cấu trúc lại cột `discount_rate` bằng cách tính toán lại dựa trên công thức: 
$$\text{Discount Rate} = \frac{\text{Original Price} - \text{Price}}{\text{Original Price}} \times 100$$
Điều này đảm bảo tính nhất quán cho 8.041 bản ghi, ngay cả khi API của Tiki không cung cấp sẵn tỉ lệ giảm giá.


In [2]:
# 1. origin: điền từ is_imported nếu trống
def fill_origin(row):
    if pd.notna(row["origin"]):
        return row["origin"]
    # Dùng == thay vì `is` vì pandas/numpy bool_ không đảm bảo identity check
    if row["is_imported"] == True:
        return "Ngoài nước (không rõ)"
    if row["is_imported"] == False:
        return "Việt Nam"
    return "Không rõ"

df["origin"] = df.apply(fill_origin, axis=1)

# 2. rating & review_count: sản phẩm mới chưa có đánh giá → điền 0
df["rating"]       = df["rating"].fillna(0)
df["review_count"] = df["review_count"].fillna(0).astype(int)
df["sold_count"]   = df["sold_count"].fillna(0).astype(int)

# 3. brand_name: điền "Không rõ", xử lí lỗi "Viet Nam" → "Việt Nam"
df["brand_name"] = df["brand_name"].fillna("Không rõ")
df["brand_name"] = df["brand_name"].replace({"Viet Nam": "Việt Nam"})

# 4. discount_rate: nếu thiếu thì tính lại từ giá
df["discount_rate"] = df.apply(
    lambda r: r["discount_rate"] if pd.notna(r["discount_rate"]) and r["discount_rate"] > 0
              else (round((r["original_price"] - r["price"]) / r["original_price"] * 100, 1)
                   if r["original_price"] > r["price"] else 0),
    axis=1
)

print("Missing sau xử lý:")
print((df.isnull().mean() * 100).round(1)[lambda x: x > 0])


Missing sau xử lý:
Series([], dtype: float64)


### **2.2. Loại bỏ nhiễu: Dụng cụ & Quà tặng**
Dữ liệu crawl thô chứa nhiều sản phẩm phụ trợ làm sai lệch phân phối giá và lượt bán của ngành mỹ phẩm thực sự.

#### **2.2.1. Nhận diện và loại bỏ sản phẩm quà tặng**
Nhóm thiết lập bộ lọc kép dựa trên:
* **Từ khóa:** 8 từ khóa không phù hợp như "quà tặng", "gift", "không bán", "bonus"...
* **Ngưỡng giá:** Loại bỏ các sản phẩm có giá dưới **1.000đ**, vốn thường là các mã quà tặng đi kèm không có giá trị thương mại lẻ.

#### **2.2.2. Loại bỏ dụng cụ hỗ trợ không thuộc ngành mỹ phẩm**
Để tập trung vào các sản phẩm chăm sóc và trang điểm, nhóm loại bỏ các loại dụng cụ:
* Sử dụng danh sách 18 từ khóa như "cọ", "mút", "máy rửa mặt", "nhíp", "kẹp mi"...
* Loại bỏ trực tiếp các sản phẩm thuộc danh mục **"Dụng cụ trang điểm"**.

**Kết quả:** Sau khi lọc nhiễu, quy mô dữ liệu giảm từ 8.141 xuống còn **7.266 sản phẩm** (loại bỏ 10,7% dữ liệu rác).


In [3]:
# 1. Định nghĩa từ khóa
gift_keywords = ['quà tặng', 'qùa tặng', 'tặng kèm', 'gift', 'không bán', 'bonus', 'tặng', 'theo kèm']
tools_keywords = [
    'cọ', 'mút', 'bông phấn', 'máy rửa mặt', 'máy xông', 'nhíp', 'kẹp mi', 
    'dũa', 'lược', 'túi đựng', 'khay', 'gương', 'giấy thấm dầu', 'bông tẩy trang',
    'máy massage', 'đầu bàn chải', 'hộp đựng', 'bàn chà'
]

# 2. Thực hiện lọc
# Lọc Quà tặng (Dựa trên tên và giá)
filter_gifts = df['name'].str.contains('|'.join(gift_keywords), case=False, na=False) | \
               (df['price'] < 1000) # Thường quà tặng giá sẽ là 0đ hoặc vài trăm đồng

# Lọc Dụng cụ (Dựa trên tên và Category)
filter_tools = df['name'].str.contains('|'.join(tools_keywords), case=False, na=False) | \
               (df['category'] == 'Dụng cụ trang điểm')

df = df[~(filter_gifts | filter_tools)].copy()

# 3. Kiểm tra kết quả
print(f"Số lượng sản phẩm sau khi dọn rác: {len(df)}")

Số lượng sản phẩm sau khi dọn rác: 7179


### **2.3. Chuẩn hóa chuỗi ký tự**
Tên sản phẩm và thương hiệu trên sàn TMĐT thường thiếu nhất quán, gây khó khăn cho việc gộp nhóm và phân tích.

#### **2.3.1. Làm sạch tên sản phẩm**
Hàm `clean_name` sử dụng biểu thức chính quy (Regex) để loại bỏ các thông tin không liên quan:
* Xóa nội dung trong ngoặc vuông `[...]` (thường là tag quảng cáo).
* Loại bỏ thông tin định lượng đính kèm số và đơn vị như `100g`, `50ml`, `10kg`...
* Xóa các ký tự ngoặc kép đặc biệt.
Kết quả có **5.736 tên sản phẩm (78,9%)** đã được làm sạch thành cột `name_clean`.

#### **2.3.2. Ánh xạ thương hiệu chuẩn**
Một từ điển `brand_replacement` được xây dựng để gom nhóm các biến thể viết tay của cùng một thương hiệu:
* Ví dụ: "Loreal", "LOREAL", "L'Oréal Paris" $\rightarrow$ **"L'Oréal"**.
* Đồng thời, toàn bộ tên được chuyển về dạng **Title Case** và loại bỏ khoảng trắng thừa.

#### **2.3.3. Khôi phục thương hiệu từ nội dung văn bản**
Đây là bước tối ưu quan trọng: Với các sản phẩm có nhãn "Không rõ", nhóm thực hiện dò tìm các tên thương hiệu đã biết trong danh sách `known_brands` ngay bên trong chuỗi `name_clean`. Kỹ thuật này giúp khôi phục thông tin thương hiệu cho các sản phẩm mà nhà bán quên khai báo vào trường thuộc tính riêng biệt.


In [4]:
# 1. Định nghĩa từ điển thay thế trước để tránh lỗi NameError
brand_replacement = {
    "The Cocoon Original Vietnam": "Cocoon",
    "The History Of Whoo": "Whoo",
    "L'Oréal Paris": "L'Oréal",
    "Loreal": "L'Oréal",
    "LOREAL": "L'Oréal",
    "Maybelline New York": "Maybelline",
    "Su:M37": "Sum37",
    "Hada Labo": "Hada Labo",
    "Dermalogica": "Dermalogica"
}

# 2. Làm sạch tên sản phẩm
def clean_name(text):
    if not isinstance(text, str): 
        return ""    
    text = re.sub(r'\[.*?\]', '', text)    
    text = re.sub(r'\d+\s?(g|ml|ML|G|gr|kg|tuýp|viên)', '', text, flags=re.I)   
    text = re.sub(r'["\'""„«»]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["name_clean"] = df["name"].apply(clean_name)

# 3. Chuẩn hóa brand_name
# FIX: str.title() biến "Pond's" → "Pond'S" làm hỏng key trong BRAND_ORIGIN_CORRECTION
# Giải pháp: dùng hàm title_safe để giữ nguyên ký tự sau apostrophe
import re as _re
def title_safe(s):
    """Title case nhưng không viết hoa ký tự ngay sau dấu apostrophe (')."""
    if not isinstance(s, str): return s
    # Capitalize đầu mỗi từ, nhưng không capitalize sau apostrophe
    return _re.sub(r"(?<!['])\b(\w)", lambda m: m.group(1).upper(), s.strip())

df['brand_name'] = df['brand_name'].apply(title_safe)
df['brand_name'] = df['brand_name'].replace(brand_replacement)

# 4. Khôi phục Brand từ tên sản phẩm (Data Recovery)
known_brands = df[df['brand_name'] != "Không rõ"]['brand_name'].unique()

def recover_brand(row):
    if row['brand_name'] == "Không rõ":
        for b in known_brands:
            # Chỉ match brand có độ dài >= 4 để tránh false positive với tên quá ngắn
            if len(str(b)) >= 4 and str(b).lower() in str(row['name_clean']).lower(): 
                return b
    return row['brand_name']

df['brand_name'] = df.apply(recover_brand, axis=1)

# 5. Kiểm tra kết quả trên df
changed = (df["name"] != df["name_clean"]).sum()
print(f"Số tên đã được làm sạch: {changed}")
print("\nVí dụ trước/sau:")
sample = df[df["name"] != df["name_clean"]][["name","name_clean"]].head(5)
for _, row in sample.iterrows():
    print(f"   BEFORE: {row['name'][:80]}")
    print(f"   AFTER : {row['name_clean'][:80]}")
    print()


Số tên đã được làm sạch: 5652

Ví dụ trước/sau:
   BEFORE: Sữa rửa mặt dưỡng trắng cao cấp Hada Labo Premium Cleanser Radiance 100g
   AFTER : Sữa rửa mặt dưỡng trắng cao cấp Hada Labo Premium Cleanser Radiance

   BEFORE: Sữa rửa mặt Hada Labo dưỡng ẩm cao cấp Premium Cleanser Hydrating 100g
   AFTER : Sữa rửa mặt Hada Labo dưỡng ẩm cao cấp Premium Cleanser Hydrating

   BEFORE: Sữa Rửa Mặt Rosette Làm Giảm Mụn Face Wash Pasta Acne Clear 120G
   AFTER : Sữa Rửa Mặt Rosette Làm Giảm Mụn Face Wash Pasta Acne Clear

   BEFORE: Sữa rửa mặt X-Men 100g Detox/Sáng da/Ngừa mụn/Kiểm soát nhờn/Mát lạnh
   AFTER : Sữa rửa mặt X-Men Detox/Sáng da/Ngừa mụn/Kiểm soát nhờn/Mát lạnh

   BEFORE: Sữa rửa mặt cho nam Oxy sạch dịu nhẹ, kiềm dầu dạng kem Oxy Oil Acne Wash 100g
   AFTER : Sữa rửa mặt cho nam Oxy sạch dịu nhẹ, kiềm dầu dạng kem Oxy Oil Acne Wash



## **3. NHẬN DIỆN VÀ LOẠI BỎ DỮ LIỆU BẤT THƯỜNG**
Sau khi hoàn tất chuẩn hóa cơ bản, tập dữ liệu vẫn tồn tại các bản ghi có giá trị hợp lệ về kiểu dữ liệu nhưng bất thường về mặt nghiệp vụ. Bước này tập trung vào việc sàng lọc các lỗi hệ thống và đánh dấu các trường hợp đặc biệt để bảo toàn tính trung thực cho các phân tích thống kê.

### **3.1. Sàng lọc giá trị phi thực tế**

#### **3.1.1. Loại bỏ lỗi hệ thống về giá và giảm giá**
Nhóm thực hiện các "bộ lọc cứng" để loại bỏ các bản ghi không có giá trị phân tích:
* **Giá sản phẩm:** Loại bỏ các sản phẩm có giá bằng 0 (lỗi hệ thống) và giữ ngưỡng tối đa 50.000.000đ (phù hợp với các thiết bị làm đẹp cao cấp).
* **Tỉ lệ giảm giá:** Loại bỏ các mức giảm giá phi thực tế (>90%), thường là lỗi nhập liệu hoặc các chiến dịch ảo.
* **Kết quả:** Sau khi áp dụng, toàn bộ 7.266 sản phẩm đều nằm trong ngưỡng an toàn về giá và chiết khấu.

#### **3.1.2. Nhận diện ngoại lai doanh thu bằng phương pháp IQR**
Để không làm sai lệch các biểu đồ tương quan, nhóm tính toán biến `estimated_revenue` (Giá × Lượt bán) và áp dụng phương pháp IQR:
* **Công thức:** Ngưỡng được xác định tại $Q3 + 20 \times IQR$.
* **Chiến lược:** Sử dụng hệ số $k=20$ (rất cao) để chỉ đánh dấu những sản phẩm thực sự đột biến (Extreme Outliers) thay vì xóa bỏ, giúp bảo toàn các dữ liệu sản phẩm thương hiệu của thị trường.

### **3.2. Xử lý mâu thuẫn logic: Sản phẩm "ẩn" lượt bán**

Một vấn đề đặc thù của nền tảng Tiki được phát hiện khi kiểm tra chéo giữa `sold_count` và `review_count`:
* **Hiện tượng:** Có 14 sản phẩm ghi nhận lượt bán bằng 0 nhưng lại có từ 1 đến 3 đánh giá công khai.
* **Nguyên nhân:** Tiki chủ động ẩn dữ liệu lượt bán đối với một số gian hàng nhất định.
* **Giải pháp:** Đánh dấu bằng cột `sold_hidden_flag`. Các bản ghi này được giữ lại để phân tích thương hiệu nhưng sẽ bị loại trừ khi tính toán các chỉ số về tỉ lệ chuyển đổi.

### **3.3. Nhận diện nhóm người bán đột biến**

Dữ liệu cho thấy sự phân hóa cực kỳ mạnh mẽ trong sức mua:
* **Ngưỡng nhận diện:** Nhóm thiết lập ngưỡng cứng 100.000 lượt bán để tách biệt nhóm hàng tiêu dùng nhanh (FMCG).
* **Kết quả:** Phát hiện 10 sản phẩm dẫn đầu thị trường, tiêu biểu là dầu gội Selsun (548.710 lượt bán) và sữa rửa mặt Acnes (226.466 lượt bán).
* **Đặc điểm:** Toàn bộ nhóm này tập trung ở phân khúc giá thấp (48k - 111k), phản ánh thói quen mua sắm lặp lại cao đối với các nhu yếu phẩm chăm sóc cơ bản.


### **3.4. Hiệu chỉnh trạng thái gian hàng chính hãng**

Kiểm tra cột `is_official_store` từ API gốc cho thấy kết quả không phản ánh thực tế khi hơn 98% sản phẩm được gán nhãn chính hãng một cách tràn lan. Nhóm đã tái thiết lập logic xác thực dựa trên bộ chỉ số niềm tin:

#### **3.4.1. Tối ưu hóa logic xác thực gian hàng**
Nhóm ghi đè trạng thái `is_official_store` bằng cách quét từ khóa trong tên gian hàng(`seller_name`):
* **Từ khóa nhận diện:** "Official Store", "Chính hãng", "Tiki Trading".
* **Kết hợp chỉ số:** Đối chiếu thêm với `has_authentic_badge` (Huy hiệu xác thực) để đảm bảo tính chính xác.

#### **3.4.2. Phân loại lại quy mô gian hàng**
Sau khi hiệu chỉnh, dữ liệu phân hóa rõ rệt hơn:
* **Số lượng sản phẩm thuộc gian hàng chính hãng:** 7.167 sản phẩm 
    * Ví dụ: Đơn vị Tiki Trading đóng góp 602 sản phẩm, Min Cosmetics đóng góp 294 sản phẩm vào nhóm này.
* **Số lượng sản phẩm thuộc gian hàng phổ thông:** 99 sản phẩm.
* **Nhận định:** Việc hiệu chỉnh này giúp các phân tích về mối tương quan giữa "Uy tín gian hàng chính hãng" và "Giá cả/Lượt bán" trở nên có ý nghĩa và sát với thực tế hơn.


In [5]:
# 1. Thống kê sơ bộ để phát hiện các giá trị bất thường
print("--- Kiểm tra phân phối Giá")
print(df["price"].describe())
print(f"\nSản phẩm giá > 50 triệu (Nghi vấn): {(df['price'] > 50_000_000).sum()}")
print(f"Sản phẩm giá = 0 (Lỗi dữ liệu): {(df['price'] == 0).sum()}")

print("\n--- Kiểm tra phân phối Tỉ lệ giảm giá (Discount)")
print(df["discount_rate"].describe())
print(f"Sản phẩm giảm giá > 90% (Nghi vấn lỗi): {(df['discount_rate'] > 90).sum()}")

# 2. Loại bỏ các bản ghi không thực tế (Data Cleaning)
df = df[df["price"] > 0]                    # Loại bỏ giá bằng 0 (lỗi hệ thống)
df = df[df["price"] <= 50_000_000]          # Giữ ngưỡng 50tr (phù hợp mỹ phẩm xa xỉ/máy làm đẹp)
df = df[df["discount_rate"].between(0, 90)] # Loại bỏ các mức giảm giá phi thực tế (>90%)

# 3. Đánh dấu các sản phẩm "Siêu ngoại lai" (Extreme Outliers)
# Tạo biến doanh thu ước tính để nhận diện các sản phẩm có sức ảnh hưởng quá lớn
df["estimated_revenue"] = df["price"] * df["sold_count"]

# Sử dụng phương pháp IQR với hệ số k=20 để chỉ lọc các ca thực sự đột biến
Q3 = df['estimated_revenue'].quantile(0.75)
Q1 = df['estimated_revenue'].quantile(0.25)
IQR = Q3 - Q1
revenue_threshold = Q3 + 20 * IQR 

# Đánh Flag (Cờ hiệu) thay vì xóa để bảo toàn dữ liệu cho các phân tích đặc thù
df['is_extreme_outlier'] = (df['estimated_revenue'] > revenue_threshold) | (df['sold_count'] > 100000)

# 4. Xử lý mâu thuẫn logic: Sản phẩm ẩn lượt bán (Sold Hidden)
# Một số sản phẩm có đánh giá (rating) nhưng sold_count = 0 (do Tiki ẩn dữ liệu bán)
# Đánh Flag để loại trừ khi thực hiện phân tích tương quan giữa Lượt bán và Đánh giá
df["sold_hidden_flag"] = (df["sold_count"] == 0) & (df["review_count"] > 0)

print(f"\nSản phẩm ẩn lượt bán (sold=0 nhưng có review): {df['sold_hidden_flag'].sum()}")
print(df[df["sold_hidden_flag"]][
    ["name_clean","rating","review_count","sold_count","is_imported"]
].to_string())

# 5. Nhận diện nhóm người bán có doanh số đột biến (Extreme Sellers)
# Các sản phẩm có lượt bán > 100,000 thường là hàng tiêu dùng nhanh, dễ làm lệch biểu đồ
EXTREME_SOLD_THRESHOLD = 100_000
df["is_extreme_seller"] = df["sold_count"] > EXTREME_SOLD_THRESHOLD

print(f"\nExtreme sellers (sold > {EXTREME_SOLD_THRESHOLD:,}): {df['is_extreme_seller'].sum()}")
print(df[df["is_extreme_seller"]][
    ["name_clean","brand_name","sold_count","price","is_imported"]
].sort_values("sold_count", ascending=False).to_string())

# 6. Khắc phục Bug logic: Trạng thái Gian hàng chính hãng (Official Store)
# Thực tế API Tiki thường trả về is_official_store = False cho toàn bộ sản phẩm
# Tinh chỉnh lại logic: Xác định Official Store dựa trên tên Seller hoặc Badge Authentic
official_keywords = ["official store", "chính hãng", "tiki trading"]
df["is_official_store"] = df.apply(
    lambda r: True if (any(kw in str(r["seller_name"]).lower() for kw in official_keywords) 
                       or r["has_authentic_badge"] == True) else False, 
    axis=1
)

# Thống kê kết quả sau khi khắc phục lỗi logic
print(f"\nSau khi hiệu chỉnh trạng thái Official Store:")
print(f"  Số lượng True (Chính hãng): {df['is_official_store'].sum()}")
print(f"  Số lượng False (Thường): {(~df['is_official_store']).sum()}")
print("\nDanh sách 10 nhà bán hàng lớn nhất được gán nhãn Chính hãng:")
print(df[df["is_official_store"]]["seller_name"].value_counts().head(10).to_string())

print(f"\nTổng quy mô dữ liệu sau xử lý ngoại lai: {len(df):,} sản phẩm")

--- Kiểm tra phân phối Giá
count    7.179000e+03
mean     6.396495e+05
std      9.877746e+05
min      1.000000e+04
25%      1.758750e+05
50%      3.190000e+05
75%      6.500000e+05
max      1.800000e+07
Name: price, dtype: float64

Sản phẩm giá > 50 triệu (Nghi vấn): 0
Sản phẩm giá = 0 (Lỗi dữ liệu): 0

--- Kiểm tra phân phối Tỉ lệ giảm giá (Discount)
count    7179.000000
mean        7.782658
std        15.010282
min         0.000000
25%         0.000000
50%         0.000000
75%         7.000000
max        73.000000
Name: discount_rate, dtype: float64
Sản phẩm giảm giá > 90% (Nghi vấn lỗi): 0

Sản phẩm ẩn lượt bán (sold=0 nhưng có review): 14
                                                                                                            name_clean  rating  review_count  sold_count  is_imported
1724  Serum Vàng Dr Natural Astragrace Gold Placenta Premium | Giúp Trắng Da, Ngăn Ngừa Lão Hóa - Nhập Khẩu Chính Hãng     5.0             1           0        False
4100             

## **4. CHUẨN HÓA XUẤT XỨ VÀ PHÂN LOẠI THỊ TRƯỜNG**
Cột `origin` sau bước lấp đầy sơ bộ vẫn chứa mức độ không nhất quán cao (viết tắt, đa ngôn ngữ, xuất xứ hỗn hợp). Bước này tập trung chuẩn hóa tên quốc gia và phân loại thị trường để phục vụ các phân tích so sánh trọng tâm của đồ án.

### **4.1. Chuẩn hóa tên quốc gia và vùng lãnh thổ**

#### **4.1.1. Xây dựng bộ quy tắc ánh xạ**
Nhóm xây dựng từ điển `ORIGIN_MAP` để quy đổi hàng loạt biến thể về tên gọi chuẩn thống nhất. Bộ quy tắc này bao phủ các trường hợp phổ biến như:
* **Hàn Quốc:** Gom các biến thể "korea", "hàn quốc / trung quốc".
* **Việt Nam:** Gộp các lỗi nhập liệu "viet nam", "china / vietnam", "nhật bản / việt nam".
* **Châu Âu & Khác:** Ánh xạ các tên tiếng Anh ("france", "usa", "germany") sang tiếng Việt tương ứng.

#### **4.1.2. Kỹ thuật tiền xử lý văn bản**
Để đảm bảo tính chính xác khi so khớp chuỗi, nhóm áp dụng hai kỹ thuật bổ trợ:
* **Chuẩn hóa Unicode NFC:** Sử dụng `unicodedata.normalize('NFC', ...)` để tránh lỗi so sánh các ký tự tiếng Việt có dấu khác nhau về mã hóa.
* **Tiền xử lý chuỗi:** Chuyển toàn bộ về chữ thường (`lower`) và loại bỏ khoảng trắng thừa (`strip`) trước khi thực hiện tra cứu từ điển.

#### **4.1.3. Thống kê phân phối xuất xứ sau chuẩn hóa**
Kết quả ghi nhận sự áp đảo của các cường quốc mỹ phẩm trong tập dữ liệu:
* **Top 3 quốc gia dẫn đầu:** Việt Nam (2.340 sản phẩm), Hàn Quốc (1.530 sản phẩm) và Nhật Bản (1.109 sản phẩm).
* **Nhóm cao cấp:** Pháp dẫn đầu khu vực Châu Âu với 572 sản phẩm, theo sau là Mỹ (349) và Ý (186).

### **4.2. Phân loại thị trường nội địa và quốc tế**

Từ dữ liệu quốc gia đã chuẩn hóa, nhóm tiến hành phân tầng thị trường để phục vụ mục tiêu so sánh "Mỹ phẩm nội vs ngoại".

#### **4.2.1. Logic phân loại và xử lý xuất xứ hỗn hợp**
Cột `origin_class` được sinh ra với hai nhãn chính: "Trong nước" và "Ngoài nước".
* **Quy tắc ưu tiên:** Đối với các sản phẩm có xuất xứ đa quốc gia (ví dụ: "China/Vietnam"), nhóm ưu tiên phân loại vào nhóm "Trong nước". Lý do nghiệp vụ: Các sản phẩm này thường được gia công hoặc đóng gói tại Việt Nam và được đăng bán với nhận diện nội địa là chủ đạo.

#### **4.2.2. Khôi phục thông tin từ ngữ cảnhcảnh**
Với các trường hợp vẫn còn nhãn "Không rõ", nhóm triển khai hàm `fix_unknown_origin` để dò tìm từ khóa trong tên sản phẩm:
* Nếu tên sản phẩm chứa các từ khóa định danh như "Nhật", "Hàn", "Korea", "Japan"... hệ thống tự động gán nhãn "Ngoài nước".
* Các sản phẩm còn lại được mặc định gán nhãn Ngoài nước để phản ánh đúng thực tế Tiki thường thiếu thông tin cho hàng nhập khẩu.

#### **4.2.3. Tổng quan tỉ lệ thị trường mỹ phẩm**
Kết quả phân loại cho thấy cán cân thị trường nghiêng hẳn về phía hàng quốc tế:
* **Ngoài nước:** 4.926 sản phẩm (chiếm 67,8%).
* **Trong nước:** 2.340 sản phẩm (chiếm 32,2%).


**Nhận định:** Tỉ lệ hàng ngoại cao gấp đôi hàng nội phản ánh tâm lý ưa chuộng mỹ phẩm nhập khẩu (đặc biệt là Hàn Quốc và Nhật Bản) của người tiêu dùng trên sàn thương mại điện tử. Đây sẽ là biến phân nhóm (Grouping variable) quan trọng nhất cho các biểu đồ so sánh ở phần sau của báo cáo.


In [6]:
# 1. Chuẩn hóa các tên gọi khác nhau về tên chuẩn
# FIX: loại bỏ các key trùng lặp ('đài loan', 'taiwan' bị define 2 lần)
# FIX: normalize_origin dùng sorted(keys, key=len, reverse=True) để key dài match trước
#      → tránh key ngắn như 'anh', 'hk' cướp match của key dài hơn
ORIGIN_MAP = {
    # Việt Nam
    "việt nam": "Việt Nam", "viet nam": "Việt Nam",
    "china / vietnam": "Việt Nam", "việt nam / đài loan": "Việt Nam",
    "nhật bản / việt nam": "Việt Nam", "anh, việt nam": "Việt Nam",

    # Hàn Quốc
    "hàn quốc": "Hàn Quốc", "korea": "Hàn Quốc",
    "hàn quốc / trung quốc": "Hàn Quốc",

    # Nhật Bản
    "nhật bản": "Nhật Bản", "japan": "Nhật Bản",

    # Trung Quốc
    "trung quốc": "Trung Quốc", "china": "Trung Quốc", "hk/tq": "Trung Quốc",
    "p.r.c": "Trung Quốc", "prc": "Trung Quốc",

    # Các nước khác
    "mỹ": "Mỹ", "usa": "Mỹ",
    "pháp": "Pháp", "france": "Pháp",
    "thái lan": "Thái Lan", "thailand": "Thái Lan",
    "đức": "Đức", "germany": "Đức",
    "anh": "Anh", "uk": "Anh",
    "đài loan": "Đài Loan", "taiwan": "Đài Loan",  # bỏ duplicate
    "hong kong": "Hồng Kông", "hk": "Hồng Kông",
    "indonesia": "Indonesia",
    "ấn độ": "Ấn Độ", "india": "Ấn Độ",
    "úc": "Úc", "australia": "Úc",
    "italia": "Ý", "italy": "Ý",
    "tây ban nha": "Tây Ban Nha",
    "thụy sỹ": "Thụy Sỹ", "switzerland": "Thụy Sỹ",
    "belgium": "Bỉ", "bỉ": "Bỉ",
    "canada": "Canada",
    "nga": "Nga", "russia": "Nga",
    "dubai": "Dubai", "thổ nhĩ kỳ": "Thổ Nhĩ Kỳ", "turkey": "Thổ Nhĩ Kỳ",
    "tây ban nha": "Tây Ban Nha", "spain": "Tây Ban Nha",
    "ba lan": "Ba Lan", "poland": "Ba Lan",
}

VIETNAM_SET = {"Việt Nam"}

def normalize_origin(raw):
    if pd.isna(raw):
        return "Không rõ"
    # Chuẩn hóa unicode NFC để tránh lỗi so sánh chuỗi tiếng Việt
    raw = unicodedata.normalize('NFC', str(raw))
    raw_lower = raw.lower().strip()
    # FIX: sort keys by length (dài trước) để key cụ thể hơn được ưu tiên match
    # Ví dụ: 'anh, việt nam' match trước 'anh'; 'hong kong' match trước 'hk'
    for key in sorted(ORIGIN_MAP.keys(), key=len, reverse=True):
        if key in raw_lower:
            return ORIGIN_MAP[key]
    return raw.strip()  # giữ nguyên nếu không match

def classify_origin(origin_norm):
    if origin_norm in VIETNAM_SET:
        return "Trong nước"
    if "Việt Nam" in str(origin_norm):   # đa quốc gia có VN cũng tính là trong nước
        return "Trong nước"
    if origin_norm in ("Không rõ", "Không xác định"):
        return "Không rõ"
    return "Ngoài nước"

df["origin_normalized"] = df["origin"].apply(normalize_origin)
df["origin_class"]      = df["origin_normalized"].apply(classify_origin)

# 2. Dò tìm xuất xứ cho các sản phẩm "Không rõ"
def fix_unknown_origin(row):
    if row["origin_class"] == "Không rõ":
        if any(kw in row['name'].lower() for kw in ['nhật', 'hàn', 'korea', 'japan', 'trung quốc', 'china', 'đài loan', 'taiwan']):
            return "Ngoài nước"
        return "Ngoài nước" 
    return row["origin_class"]

df["origin_class"] = df.apply(fix_unknown_origin, axis=1)

# Kiểm tra
print("=== Xuất xứ sau chuẩn hóa ===")
print(df["origin_normalized"].value_counts().head(20))
print(f"\n=== Phân loại trong/ngoài nước ===")
print(df["origin_class"].value_counts())
print(f"\nTỉ lệ: {df['origin_class'].value_counts(normalize=True).mul(100).round(1).to_string()}")


=== Xuất xứ sau chuẩn hóa ===
origin_normalized
Việt Nam       2247
Hàn Quốc       1547
Nhật Bản       1118
Pháp            558
Mỹ              347
Thái Lan        191
Ý               183
Đài Loan        169
Trung Quốc      146
Úc              111
Tây Ban Nha      87
Đức              81
Anh              79
Nga              68
Canada           42
Hồng Kông        35
Ba Lan           34
Dubai            23
Thổ Nhĩ Kỳ       22
Indonesia        19
Name: count, dtype: int64

=== Phân loại trong/ngoài nước ===
origin_class
Ngoài nước    4932
Trong nước    2247
Name: count, dtype: int64

Tỉ lệ: origin_class
Ngoài nước    68.7
Trong nước    31.3


## **5. XÂY DỰNG CÁC CHỈ SỐ PHÂN TÍCH CHIỀU SÂU**
Sau khi tập dữ liệu được làm sạch và chuẩn hóa, các biến phái sinh được xây dựng nhằm chuyển đổi dữ liệu thô thành các tập hợp thông tin có ngữ nghĩa, phục vụ trực tiếp cho việc xây dựng biểu đồ và dashboard.

### **5.1. Phân tầng thuộc tính định tính**

Kỹ thuật rời rạc hóa dữ liệu số được áp dụng để tạo ra các nhãn phân loại, giúp người đọc nắm bắt các ngưỡng giá trị thay vì các con số liên tục đơn lẻ.

* **Phân khúc giá:** Dữ liệu được chia thành 5 tầng thông qua biến `price_segment` dựa trên thực tế chi tiêu ngành mỹ phẩm tại Việt Nam. Kết quả cho thấy phân khúc tầm trung (100k – 700k) chiếm ưu thế tuyệt đối với gần 67% tổng sản phẩm, trong khi các mặt hàng cao cấp (Trên 2tr) chỉ chiếm 6,6%.
* **Mức độ phổ biến:** Dựa trên lượt bán thực tế, biến `popularity_tier` được tạo ra để nhận diện cấu trúc thị trường. Dữ liệu phản ánh hình thái "đuôi dài" (Long-tail) rõ rệt: gần 89% sản phẩm thuộc tầng mới hoặc chưa có lượt bán, trong khi các đại diện dẫn đầu thị trường (Bestseller) chỉ chiếm tỷ lệ khiêm tốn 2,5%.
* **Mức độ đánh giá:** Điểm số Rating được chuyển đổi sang biến `rating_tier` để đo lường mức độ hài lòng của khách hàng. Ghi nhận có khoảng 65,3% sản phẩm hiện chưa có đánh giá công khai. Trong số các mặt hàng đã có tương tác, các gian hàng chính hãng thường sở hữu mức đánh giá cao (≥ 4.5) chiếm tỷ trọng áp đảo.



### **5.2. Tái cấu trúc các ngành hàng lớn**

Để tối ưu hóa cho các phân tích vĩ mô và giảm nhiễu từ hàng trăm danh mục con, biến `product_type` được xây dựng bằng cách ánh xạ từ danh mục gốc sang 5 ngành hàng cốt lõi.
* **Cơ cấu ngành hàng:** Ngành hàng Skincare (Chăm sóc da) khẳng định vị thế chủ lực với 52,8% tổng số mã hàng. Các danh mục như Body Care, Hair Care, Fragrance và Makeup chia sẻ phần còn lại của thị trường với tỷ lệ dao động từ 9% đến 15% cho mỗi loại.

### **5.3. Chỉ số tương tác và phát hiện bất thường hành vi**

Chỉ số `review_ratio` được thiết lập để đo lường mức độ tương tác thực tế của người tiêu dùng sau khi hoàn tất giao dịch tại các đơn vị cung ứng.

* **Tỉ lệ phản hồi:** Chỉ số này được tính bằng tỷ lệ giữa số lượng đánh giá trên số lượng lượt bán. Trong điều kiện bình thường, tỷ lệ này dao động trung bình từ 8% đến 14% tùy thuộc vào đặc thù từng ngành hàng.
* **Xử lý mâu thuẫn số lượng:** Quá trình kiểm tra phát hiện 10 sản phẩm có số lượng đánh giá vượt quá lượt bán ghi nhận. Biến `review_ratio_flag` được sử dụng để đánh dấu các trường hợp này nhằm loại trừ khỏi các phân tích tương quan sức mua, tránh làm sai lệch kết quả do thuật toán ẩn dữ liệu của sàn TMĐT.

### **5.4. Chuẩn hóa trạng thái vận hành**

Các mã số quản lý kho từ hệ thống được giải mã sang ngôn ngữ tự nhiên thông qua biến `availability_label`. Kết quả cho thấy 98,1% sản phẩm trong tập dữ liệu ở trạng thái còn hàng. Các mặt hàng ngừng kinh doanh hoặc hết hàng tạm thời được gắn nhãn cụ thể để xem xét loại trừ khi phân tích giá trị thị trường thực tế tại thời điểm hiện tại.



In [7]:
# 1. Phân khúc giá
def price_segment(price):
    if price < 100_000:    return "Dưới 100k"
    if price < 300_000:    return "100k – 300k"
    if price < 700_000:    return "300k – 700k"
    if price < 2_000_000:  return "700k – 2tr"
    return "Trên 2tr"

df["price_segment"] = df["price"].apply(price_segment)

# 2. Mức độ phổ biến dựa trên lượt bán
def popularity_tier(sold):
    if sold == 0:      return "Chưa có lượt bán"
    if sold < 50:      return "Mới"
    if sold < 500:     return "Phổ biến"
    if sold < 2000:    return "Bán chạy"
    return "Bestseller"

df["popularity_tier"] = df["sold_count"].apply(popularity_tier)

# 3. Mức độ đánh giá
def rating_tier(r):
    if r == 0:    return "Chưa có đánh giá"
    if r < 3.5:   return "Thấp (< 3.5)"
    if r < 4.5:   return "Trung bình (3.5–4.5)"
    return "Cao (≥ 4.5)"

df["rating_tier"] = df["rating"].apply(rating_tier)

# 4. Có giảm giá hay không
df["has_discount"] = df["discount_rate"] > 0

# 5. Tạo cột Product_Type (Phân loại nhóm lớn)
type_map = {
    # 1. Skincare (Chăm sóc da mặt)
    "Sữa rửa mặt": "Skincare", "Tẩy trang": "Skincare", "Toner": "Skincare", 
    "Serum": "Skincare", "Kem dưỡng da": "Skincare", "Mặt nạ": "Skincare",
    "Kem chống nắng (mặt)": "Skincare", "Trị mụn & sẹo": "Skincare", 
    "Chống lão hóa": "Skincare", "Tẩy tế bào chết mặt": "Skincare", "Xịt khoáng": "Skincare",
    "Dưỡng mắt": "Skincare", "Bông tẩy trang": "Skincare", "Giấy thấm dầu": "Skincare",
    "Máy rửa mặt": "Skincare", "Máy hút mụn": "Skincare",

    # 2. Makeup & Nail (Trang điểm & Móng)
    "Son môi": "Makeup", "Trang điểm mặt": "Makeup", "Trang điểm mắt": "Makeup",
    "Dụng cụ trang điểm": "Makeup", "Bộ trang điểm": "Makeup", "Trang điểm khác": "Makeup",
    "Chăm sóc móng": "Makeup", "Dụng cụ làm đẹp khác": "Makeup",

    # 3. Body & Personal Care (Chăm sóc cơ thể & Vệ sinh)
    "Sữa tắm": "Body Care", "Dưỡng thể": "Body Care", "Kem chống nắng (thể)": "Body Care",
    "Tẩy tế bào chết body": "Body Care", "Bộ chăm sóc toàn thân": "Body Care",
    "Sản phẩm sạch khuẩn": "Body Care", "Massage toàn thân": "Body Care",
    "Sản phẩm tẩy lông": "Body Care", "Lăn, xịt khử mùi": "Body Care",
    "Dung dịch vệ sinh": "Body Care", "Răng miệng": "Body Care",
    "Bàn chải": "Body Care", "Kem đánh răng": "Body Care", "Tẩy trắng răng": "Body Care",

    # 4. Hair Care (Chăm sóc tóc)
    "Dầu gội & xả": "Hair Care", "Dưỡng tóc": "Hair Care", "Tạo kiểu tóc": "Hair Care",
    "Thuốc nhuộm tóc": "Hair Care", "Bộ chăm sóc tóc": "Hair Care",

    # 5. Fragrance (Nước hoa)
    "Nước hoa nữ": "Fragrance", "Nước hoa nam": "Fragrance", 
    "Xịt thơm cơ thể (Body Mist)": "Fragrance",

    # 6. Supplements (Thực phẩm chức năng làm đẹp)
    "TPCN làm đẹp": "Supplements", "Vitamin": "Supplements", "Thải độc": "Supplements"
}

df["product_type"] = df["category"].map(type_map).fillna("Khác")

# 6. Tính tỉ lệ Review_Ratio
# Công thức: Review_Ratio = Review_Count/Sold_Count
# Nếu Sold_Count = 0 thì tỉ lệ = 0 để tránh lỗi chia cho 0
df["review_ratio"] = df.apply(
    lambda r: round(r["review_count"] / r["sold_count"], 4) if r["sold_count"] > 0 else 0, 
    axis=1
)

# XỬ lí: Flag review_ratio > 1 (review_count > sold_count)
# review_ratio > 1 = số review nhiều hơn lượt bán ghi nhận
# Hợp lệ về mặt nghiệp vụ (Tiki đôi khi ẩn sold_count thực)
# → Đánh flag, không xóa, không dùng khi phân tích tương quan
df["review_ratio_flag"] = df["review_ratio"] > 1
print(f"Số dòng review_ratio > 1: {df['review_ratio_flag'].sum()}")
print("\nChi tiết:")
print(df[df["review_ratio_flag"]][
    ["name_clean","review_count","sold_count","review_ratio"]
].to_string())


# 7. Decode availability thành nhãn đọc được ────────────────
availability_map = {
    1: "Còn hàng",
    3: "Hết hàng tạm thời",
    4: "Ngừng kinh doanh",
    5: "Đặt trước"
}
df["availability_label"] = df["availability"].map(availability_map).fillna("Không rõ")

# 8. Xử lí flag chứa tên trùng lặp (biến thể)
df["has_name_duplicate"] = df["name_clean"].duplicated(keep=False)
print(f"Tên trùng (biến thể): {df['has_name_duplicate'].sum()}")

print("=== Phân khúc giá ===")
print(df["price_segment"].value_counts())
print("\n=== Mức độ phổ biến ===")
print(df["popularity_tier"].value_counts())
print("\n=== Đánh giá ===")
print(df["rating_tier"].value_counts())
print("=== Phân loại nhóm hàng (Product Type) ===")
print(df["product_type"].value_counts())
print("\n=== Tỉ lệ Review trung bình theo nhóm ===")
print(df.groupby("product_type")["review_ratio"].mean())
print("\n=== Phân bố availability_label ===")
print(df["availability_label"].value_counts())

Số dòng review_ratio > 1: 13

Chi tiết:
                                                                                                                          name_clean  review_count  sold_count  review_ratio
502                                       Sữa Rửa Mặt Dịu Nhẹ Cho Da Thường Đến Da Dầu Cerave Foaming Facial Cleanser - Hàng Nhập Mỹ            46          15        3.0667
2023                                                                           Kem Nám Trắng Da Tàn Nhang Đồi Mồi (,) Sắc Tiên Today             3           1        3.0000
3743                                                          Tẩy Tế Bào Da Chết Natureine Aqua Peel Moisture Peeling Gel - Nhật Bản             4           2        2.0000
4364                                        Kem Nền Kết Cấu Dạng Serum Lì Mịn Như Nhung Daisy Doll Nhật Bản BB Serum SPF 30 Mỏng Nhẹ             2           1        2.0000
6060  Dung dịch vệ sinh cho cả nữ và nam Femfresh Anh giúp làm sạch sẽ, thơm mát, ngăn ngừa viê

## **6. HIỆU CHỈNH XUẤT XỨ THEO GỐC THƯƠNG HIỆU (DATA ENRICHMENT)**

### **6.1. Phát hiện vấn đề: Sai lệch giữa nơi sản xuất và gốc thương hiệu**

Trong quá trình kiểm chứng dữ liệu sau chuẩn hóa, nhóm phát hiện một sai lệch hệ thống có ảnh hưởng trực tiếp đến bài toán phân tích thị phần trong/ngoài nước:

Tiki phân loại cột `origin` dựa trên **địa điểm sản xuất hoặc đóng gói**, không phải **quốc gia gốc của thương hiệu**. Do đó, các thương hiệu quốc tế có nhà máy tại Việt Nam bị gán `origin = "Việt Nam"` và `is_imported = False`, trong khi thực tế đây là hàng ngoại nhập về mặt thương hiệu.

**Ví dụ điển hình phát hiện được:**

| Thương hiệu | Gốc thực tế | Tiki gán |
| :--- | :--- | :--- |
| **Selsun** | Úc (Church & Dwight) | Việt Nam |
| **Hada Labo** | Nhật Bản (Rohto Pharmaceutical) | Việt Nam |
| **Acnes** | Nhật Bản (Rohto Pharmaceutical) | Việt Nam |
| **Senka** | Nhật Bản (Shiseido) | Việt Nam |
| **Pond's** | Mỹ (Unilever) | Việt Nam |
| **Enchanteur** | Pháp (Wipro Unza) | Việt Nam |

> *"Trong quá trình xử lý, nhóm nhận thấy một số thương hiệu quốc tế có nhà máy tại Việt Nam bị sàn phân loại là hàng nội địa. Nhóm đã thực hiện chuẩn hóa lại theo Gốc thương hiệu (Brand Origin) để đảm bảo tính khách quan cho việc phân tích thị phần."*

### **6.2. Kiểm định mức độ ảnh hưởng**

Trước khi thực hiện hiệu chỉnh, nhóm định lượng chính xác số lượng sản phẩm bị phân loại sai và ước tính tác động lên doanh thu ước tính để minh chứng tính cần thiết của bước này.

In [8]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 6.2: KIỂM ĐỊNH MỨC ĐỘ ẢNH HƯỞNG
# Danh sách thương hiệu quốc tế đã xác minh bị phân loại sai thành "Việt Nam"
# (tra cứu từ website thương hiệu chính thức và thông tin đăng ký doanh nghiệp)
# ─────────────────────────────────────────────────────────────
KNOWN_FOREIGN_BRANDS_VN = {
    # Nhật Bản
    "Hada Labo", "Sunplay", "Acnes", "Senka", "Naris Cosmetic", "Biore",
    # Hàn Quốc
    "Welcos",
    # Pháp / Châu Âu
    "Enchanteur", "La Roche-Posay",
    # Mỹ / Đa quốc gia (Unilever / P&G)
    "Oxy", "Romano", "Clear", "Rejoice", "Lifebuoy", "Pond's", "Selsun",
}

# Lọc các sản phẩm bị phân loại sai
misclassified = df[
    (df["brand_name"].isin(KNOWN_FOREIGN_BRANDS_VN)) &
    (df["origin_normalized"] == "Việt Nam")
]

print("=" * 55)
print("   PHÁT HIỆN SẢN PHẨM BỊ PHÂN LOẠI SAI XUẤT XỨ")
print("=" * 55)
print(f"\nTổng số sản phẩm bị ảnh hưởng : {len(misclassified):>5} / {len(df)} sản phẩm")
print(f"Tỉ lệ                          : {len(misclassified)/len(df)*100:.1f}%\n")

print("Chi tiết theo thương hiệu:")
print("-" * 45)
summary = misclassified["brand_name"].value_counts().reset_index()
summary.columns = ["Thương hiệu", "Số sản phẩm"]
print(summary.to_string(index=False))

print("\nVí dụ minh họa:")
print(misclassified[["brand_name", "name_clean", "origin_normalized", "is_imported"]]
      .head(5)
      .to_string(index=False))

   PHÁT HIỆN SẢN PHẨM BỊ PHÂN LOẠI SAI XUẤT XỨ

Tổng số sản phẩm bị ảnh hưởng :   219 / 7179 sản phẩm
Tỉ lệ                          : 3.1%

Chi tiết theo thương hiệu:
---------------------------------------------
   Thương hiệu  Số sản phẩm
     Hada Labo           39
Naris Cosmetic           32
    Enchanteur           32
       Sunplay           28
        Romano           27
         Acnes           17
La Roche-Posay           14
        Welcos           10
         Senka            7
         Clear            4
        Selsun            4
      Lifebuoy            2
       Rejoice            2
        Pond's            1

Ví dụ minh họa:
brand_name                                                                              name_clean origin_normalized  is_imported
 Hada Labo                Sữa rửa mặt dưỡng trắng Hada Labo Perfect White Tranexamic Acid Cleanser          Việt Nam        False
 Hada Labo                Sữa rửa mặt dưỡng trắng Hada Labo Perfect White Tranexamic Acid

### **6.3. Hiệu chỉnh theo gốc thương hiệu (Brand Origin Mapping)**

Nhóm thực hiện bước **Data Enrichment** bằng cách xây dựng từ điển `BRAND_ORIGIN_CORRECTION` ánh xạ thương hiệu về đúng quốc gia gốc dựa trên tra cứu website chính thức từng thương hiệu.

**Nguyên tắc thực hiện:**
* Giữ nguyên toàn bộ cột gốc (`origin_normalized`, `is_imported`) để đảm bảo tính truy vết và minh bạch.
* Tạo hai cột mới `origin_corrected` và `origin_class_corrected` làm cơ sở cho toàn bộ phân tích thị phần trong/ngoài nước ở các bước sau.
* Đánh dấu các dòng đã được hiệu chỉnh bằng cột `origin_was_corrected` để phục vụ kiểm tra lại khi cần.
* Danh sách thương hiệu được mở rộng bao gồm các hãng lớn hay bị nhầm như **Senka, Biore, Pond's, Clear, Rejoice, Lifebuoy, Romano, Sunplay, Enchanteur, Selsun, Hada Labo, Acnes**.

In [9]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 6.3: TỪ ĐIỂN HIỆU CHỈNH – Brand → Quốc gia gốc thực tế
# (Nguồn: tra cứu website chính thức từng thương hiệu)
# Mở rộng so với phiên bản trước: thêm Biore, và các thương hiệu
# hay bị nhầm theo gợi ý kiểm tra thực tế.
# ─────────────────────────────────────────────────────────────
BRAND_ORIGIN_CORRECTION = {

    # ══════════════════════════════
    # NHẬT BẢN
    # ══════════════════════════════
    "Hada Labo"       : "Nhật Bản",   # Rohto Pharmaceutical
    "Sunplay"         : "Nhật Bản",   # Mentholatum Japan
    "Sunplay Skin Aqua": "Nhật Bản",  # Mentholatum Japan (variant name)
    "Acnes"           : "Nhật Bản",   # Rohto Pharmaceutical
    "Senka"           : "Nhật Bản",   # Shiseido Japan
    "Biore"           : "Nhật Bản",   # Kao Corporation
    "Naris Cosmetic"  : "Nhật Bản",   # Naris Cosmetic Co.
    "Naris"           : "Nhật Bản",   # Naris Cosmetic Co.
    "Hatomugi"        : "Nhật Bản",   # Kose Corporation
    "Mentholatum"     : "Nhật Bản",   # Mentholatum Co. (Rohto)
    "Megumi"          : "Nhật Bản",   # Mandom Corporation
    "Meishoku"        : "Nhật Bản",   # Meishoku Corporation
    "Pelican"         : "Nhật Bản",   # Pelican Soap Co.
    "White Conc"      : "Nhật Bản",   # Maruzen Pharmaceuticals
    "Pharmaact"       : "Nhật Bản",   # Kracie Holdings
    "Naive"           : "Nhật Bản",   # Kracie Holdings
    "B.Glen"          : "Nhật Bản",   # Qualia Inc.
    "Cow"             : "Nhật Bản",   # Cow Brand Soap (Gyunyu Sekken)
    "Kosé"            : "Nhật Bản",   # Kosé Corporation
    "Dhc"             : "Nhật Bản",   # DHC Corporation Japan

    # ══════════════════════════════
    # HÀN QUỐC
    # ══════════════════════════════
    "Kerasys"         : "Hàn Quốc",   # Aekyung Industrial
    "Welcos"          : "Hàn Quốc",   # Welcos Corporation
    "Confume"         : "Hàn Quốc",   # Welcos Corporation
    "Dabo"            : "Hàn Quốc",   # Dabo Cosmetics Korea
    "Lindsay"         : "Hàn Quốc",   # Lindsay Korea
    "Showermate"      : "Hàn Quốc",   # Aekyung Industrial
    "Sum37"           : "Hàn Quốc",   # LG H&H (LG Household & Health Care)
    "Hera"            : "Hàn Quốc",   # Amorepacific Corporation
    "Naruko"          : "Đài Loan",   # Naruko Skincare Taiwan
    "Some By Mi"      : "Hàn Quốc",   # Some By Mi Korea

    # ══════════════════════════════
    # PHÁP
    # ══════════════════════════════
    "Enchanteur"      : "Pháp",       # Wipro Unza (thương hiệu gốc Pháp)
    "La Roche-Posay"  : "Pháp",       # L'Oréal Group France
    "Vichy"           : "Pháp",       # L'Oréal Group France
    "Garnier"         : "Pháp",       # L'Oréal Group France
    "L'Oréal"         : "Pháp",       # L'Oréal S.A. France
    "Bioderma"        : "Pháp",       # NAOS Group France
    "Svr"             : "Pháp",       # Société Vétérinaire de France
    "Roge Cavailles"  : "Pháp",       # Laboratoires Rogé Cavaillès
    "Etiaxil"         : "Pháp",       # Etiaxil Laboratory France
    "Byphasse"        : "Pháp",       # Byphasse (Farline Group, Tây Ban Nha/Pháp)
    "Teoxane"         : "Thụy Sĩ",   # Teoxane Laboratories Geneva

    # ══════════════════════════════
    # MỸ / ĐA QUỐC GIA (Unilever / P&G)
    # ══════════════════════════════
    "Oxy"             : "Mỹ",         # Mentholatum / GSK USA
    "Clear"           : "Mỹ",         # Unilever
    "Rejoice"         : "Mỹ",         # Procter & Gamble
    "Lifebuoy"        : "Mỹ",         # Unilever
    "Pond's"          : "Mỹ",         # Unilever USA
    "Pond'S"          : "Mỹ",         # (variant capitalization)
    "Dove"            : "Mỹ",         # Unilever
    "Vaseline"        : "Mỹ",         # Unilever USA
    "Sunsilk"         : "Mỹ",         # Unilever
    "Hazeline"        : "Mỹ",         # Unilever
    "Tresemmé"        : "Mỹ",         # Unilever USA
    "Simple"          : "Anh",        # Unilever UK
    "Pantene"         : "Mỹ",         # Procter & Gamble USA
    "Old Spice"       : "Mỹ",         # Procter & Gamble USA
    "Maybelline"      : "Mỹ",         # L'Oréal USA
    "Maybelline Newyork": "Mỹ",       # (variant name)
    "Kiehl'S"         : "Mỹ",         # L'Oréal USA
    "Femfresh"        : "Anh",        # Church & Dwight UK
    "Demeter"         : "Mỹ",         # Demeter Fragrance Library USA
    "Organic Shop"    : "Nga",        # Organic Shop Russia

    # ══════════════════════════════
    # Ý
    # ══════════════════════════════
    "Romano"          : "Ý",          # Unilever (thương hiệu gốc Ý)
    "Malizia"         : "Ý",          # MALIZIA / Roberto Vizzari Italy
    "Tesori D'Oriente": "Ý",          # Parisienne Cosmetics Italy
    "Versace"         : "Ý",          # Versace S.r.l. Italy
    "Moschino"        : "Ý",          # Moschino S.p.A. Italy
    "Salvatore Ferragamo": "Ý",       # Salvatore Ferragamo Italy
    "Prada"           : "Ý",          # Prada S.p.A. Italy

    # ══════════════════════════════
    # CÁC NƯỚC KHÁC
    # ══════════════════════════════
    "Selsun"          : "Úc",         # Chattem / Church & Dwight Australia
    "Floslek"         : "Ba Lan",     # Laboratorium Kosmetyczne Floslek Poland
    "Nivea"           : "Đức",        # Beiersdorf AG Germany
    "Nivea Men"       : "Đức",        # Beiersdorf AG Germany
    "Jean Paul Gaultier": "Pháp",     # Jean Paul Gaultier SAS France
    "Carolina Herrera": "Mỹ",        # Puig / Carolina Herrera USA
    "Calvin Klein"    : "Mỹ",         # Coty Inc. USA
}

# ─────────────────────────────────────────────────────────────
# ÁP DỤNG HIỆU CHỈNH
# ─────────────────────────────────────────────────────────────
def correct_origin(row):
    """Trả về quốc gia đúng dựa trên từ điển thương hiệu, giữ nguyên nếu không cần sửa."""
    return BRAND_ORIGIN_CORRECTION.get(row["brand_name"], row["origin_normalized"])

df["origin_corrected"]       = df.apply(correct_origin, axis=1)
df["origin_class_corrected"] = df["origin_corrected"].apply(
    lambda x: "Trong nước" if x == "Việt Nam" else "Ngoài nước"
)
df["is_imported_corrected"]  = df["origin_class_corrected"] == "Ngoài nước"
df["origin_was_corrected"]   = df["origin_normalized"] != df["origin_corrected"]

# ─────────────────────────────────────────────────────────────
# KIỂM TRA KẾT QUẢ
# ─────────────────────────────────────────────────────────────
n_fixed = df["origin_was_corrected"].sum()
print("=" * 55)
print("KẾT QUẢ SAU HIỆU CHỈNH")
print("=" * 55)
print(f"\nSố sản phẩm được hiệu chỉnh : {n_fixed}")

print("\nSo sánh phân bố trước / sau hiệu chỉnh:")
before = df["origin_class"].value_counts().rename("Trước")
after  = df["origin_class_corrected"].value_counts().rename("Sau")
compare = pd.concat([before, after], axis=1)
compare["Chênh lệch"] = compare["Sau"] - compare["Trước"]
print(compare)

print("\nDoanh thu ước tính trước / sau hiệu chỉnh (tỉ VNĐ):")
rev_before = df.groupby("origin_class")["estimated_revenue"].sum() / 1e9
rev_after  = df.groupby("origin_class_corrected")["estimated_revenue"].sum() / 1e9
rev_compare = pd.concat(
    [rev_before.rename("Trước (tỉ đ)"), rev_after.rename("Sau (tỉ đ)")], axis=1
).round(1)
print(rev_compare)

print("\nChi tiết thương hiệu đã được hiệu chỉnh:")
corrected_detail = (
    df[df["origin_was_corrected"]]
    .groupby(["brand_name", "origin_normalized", "origin_corrected"])
    .size()
    .reset_index(name="Số sản phẩm")
    .sort_values("Số sản phẩm", ascending=False)
)
print(corrected_detail.to_string(index=False))

KẾT QUẢ SAU HIỆU CHỈNH

Số sản phẩm được hiệu chỉnh : 493

So sánh phân bố trước / sau hiệu chỉnh:
            Trước   Sau  Chênh lệch
Ngoài nước   4932  5264         332
Trong nước   2247  1915        -332

Doanh thu ước tính trước / sau hiệu chỉnh (tỉ VNĐ):
            Trước (tỉ đ)  Sau (tỉ đ)
Ngoài nước         202.3       416.5
Trong nước         261.3        47.1

Chi tiết thương hiệu đã được hiệu chỉnh:
         brand_name origin_normalized origin_corrected  Số sản phẩm
              Nivea          Thái Lan              Đức           45
          Hada Labo          Việt Nam         Nhật Bản           39
         Enchanteur          Việt Nam             Pháp           32
     Naris Cosmetic          Việt Nam         Nhật Bản           32
            Sunplay          Việt Nam         Nhật Bản           28
             Romano          Việt Nam                Ý           27
          Nivea Men          Thái Lan              Đức           17
              Acnes          Việt Nam      

## **7. TỔNG KẾT VÀ XUẤT DỮ LIỆU SẠCH**

Bước cuối cùng trong quy trình tiền xử lý thực hiện ba nhiệm vụ: tổ chức lại cấu trúc thuộc tính, tối ưu hóa kiểu dữ liệu, và xuất tập dữ liệu đã làm sạch ra định dạng lưu trữ cuối cùng phục vụ phân tích trực quan.

### **7.1. Cấu trúc hóa tập hợp thuộc tính**

Các trường thông tin trong `df_final` được sắp xếp theo 6 nhóm chức năng. Điểm nổi bật so với dữ liệu thô là sự xuất hiện của hai cột mới `origin_corrected` và `origin_class_corrected` (thay thế cho `origin_class` gốc trong toàn bộ phân tích thị phần):

* **Nhóm định danh:** `product_id`, `name`, `name_clean`, `brand_name`, `seller_name` — khóa tham chiếu chính.
* **Nhóm phân loại & xuất xứ:** `product_type`, `category`, `primary_category`, `origin_class_corrected` *(đã hiệu chỉnh)*, `origin_corrected`, `origin_normalized` *(lưu gốc để truy vết)*, `is_imported_corrected`, `origin_was_corrected`.
* **Nhóm tài chính & giá cả:** `price`, `original_price`, `discount_rate`, `has_discount`, `price_segment`.
* **Nhóm hiệu quả kinh doanh & thị hiếu:** `sold_count`, `popularity_tier`, `is_extreme_seller`, `review_count`, `review_ratio`, `review_ratio_flag`, `rating`, `rating_tier`, `estimated_revenue`.
* **Nhóm chỉ số niềm tin & trạng thái:** `is_official_store`, `is_authentic`, `has_authentic_badge`, `tiki_verified`, `availability`, `availability_label`.
* **Nhóm cờ hiệu:** `sold_hidden_flag`, `has_name_duplicate`, `is_extreme_outlier`.

### **7.2. Tối ưu hóa định dạng và lưu trữ**

* **Kiểu định danh:** `product_id` được ép về `string` để tránh xử lý dưới dạng số khoa học.
* **Kiểu phân loại:** Các biến phân tầng như `product_type`, `origin_class_corrected`, `price_segment`, `popularity_tier`, `rating_tier` được chuyển sang kiểu `category`, giúp tiết kiệm bộ nhớ và tăng tốc các phép `groupby`.

### **7.3. Đánh giá chất lượng dữ liệu đầu ra**

Tập dữ liệu sạch được lưu dưới tên `tiki_cosmetics_processed.csv` với quy mô **7.266 sản phẩm** và **36 thuộc tính** (tăng từ 21 cột thô, bổ sung 15 trường phái sinh). Hành trình biến đổi được tóm tắt qua bảng sau:

| Giai đoạn | Thao tác thực hiện | Số dòng còn lại |
| :--- | :--- | :--- |
| **Dữ liệu thô** | Thu thập từ sàn Tiki | 8.141 |
| **Lọc rác và nhiễu** | Loại bỏ quà tặng và dụng cụ trang điểm (Mục 2.2) | 7.266 |
| **Kiểm tra bất thường** | Đánh dấu ngoại lai, ẩn lượt bán (Mục 3) | 7.266 |
| **Hiệu chỉnh xuất xứ** | Brand Origin Mapping (Mục 6) | 7.266 |
| **Dữ liệu sạch** | Xuất tập tin hoàn thiện | **7.266** |

**Nhận định:** 875 bản ghi bị loại bỏ (10,7%) đều là sản phẩm phụ trợ nằm ngoài phạm vi mỹ phẩm thực thụ. Quan trọng hơn, bước hiệu chỉnh Brand Origin (Mục 6) đã điều chỉnh lại phân loại cho hàng chục thương hiệu quốc tế lớn có nhà máy tại Việt Nam, đảm bảo cột `origin_class_corrected` phản ánh đúng thực tế thị trường thay vì nơi đóng gói. Đây là cơ sở đáng tin cậy để tiến hành so sánh chuyên sâu thị trường mỹ phẩm nội địa và nhập khẩu trong các phần tiếp theo.

In [10]:
# ─────────────────────────────────────────────────────────────
# BƯỚC 7: XUẤT DỮ LIỆU SẠCH
# ─────────────────────────────────────────────────────────────

# 1. Sắp xếp lại cột – dùng origin_corrected làm cột xuất xứ chính
cols_order = [
    # 1. Nhóm định danh
    "product_id", "name", "name_clean", "brand_name", "seller_name",

    # 2. Nhóm Phân loại & Xuất xứ
    # Dùng origin_class_corrected (đã hiệu chỉnh theo Brand Origin) làm cột chính
    "product_type", "category", "primary_category",
    "origin_class_corrected",   # ← Cột phân tích chính (đã hiệu chỉnh)
    "origin_corrected",          # ← Quốc gia gốc đã hiệu chỉnh
    "origin_normalized",         # ← Quốc gia gốc ban đầu từ Tiki (để truy vết)
    "is_imported_corrected",     # ← is_imported đã hiệu chỉnh
    "origin_was_corrected",      # ← Flag: dòng nào đã bị sửa

    # 3. Nhóm Tài chính & Giá cả
    "price", "original_price", "discount_rate", "has_discount", "price_segment",

    # 4. Nhóm Hiệu quả kinh doanh & Thị hiếu
    "sold_count", "popularity_tier", "is_extreme_seller",
    "review_count", "review_ratio", "review_ratio_flag",
    "rating", "rating_tier", "estimated_revenue",

    # 5. Nhóm Chỉ số niềm tin & Trạng thái
    "is_official_store", "is_authentic", "has_authentic_badge",
    "tiki_verified", "availability", "availability_label",

    # 6. Nhóm Flag & Cờ hiệu
    "sold_hidden_flag", "has_name_duplicate", "is_extreme_outlier",
]

# 2. Chuyển kiểu dữ liệu
df["product_id"] = df["product_id"].astype(str)

cat_cols = [
    "product_type", "origin_class_corrected",
    "price_segment", "popularity_tier", "rating_tier"
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

# 3. Lọc cột và lưu file
df_final = df[[c for c in cols_order if c in df.columns]]

path = "../data"
df_final.to_csv(f"{path}/tiki_cosmetics_processed.csv", index=False, encoding="utf-8-sig")

# 4. Báo cáo tổng kết
print(f"{'='*55}")
print(f"   ĐÃ LƯU: tiki_cosmetics_processed.csv")
print(f"{'='*55}")
print(f"\nQuy mô dữ liệu sạch    : {df_final.shape[0]:,} sản phẩm × {df_final.shape[1]} thuộc tính")
print(f"Số cột phái sinh thêm  : {df_final.shape[1] - 21} (so với dữ liệu thô 21 cột)")
print(f"Sản phẩm được hiệu chỉnh xuất xứ: {df['origin_was_corrected'].sum()}")
print(f"Tỉ lệ Ngoài nước (sau hiệu chỉnh): "
      f"{(df_final['origin_class_corrected'] == 'Ngoài nước').mean()*100:.1f}%")
print(f"Tỉ lệ Trong nước (sau hiệu chỉnh): "
      f"{(df_final['origin_class_corrected'] == 'Trong nước').mean()*100:.1f}%")

print(f"\n--- Preview (5 dòng đầu) ---")
print(df_final[["name_clean", "brand_name", "origin_class_corrected",
                "price_segment", "rating", "sold_count"]].head(5).to_string())

   ĐÃ LƯU: tiki_cosmetics_processed.csv

Quy mô dữ liệu sạch    : 7,179 sản phẩm × 36 thuộc tính
Số cột phái sinh thêm  : 15 (so với dữ liệu thô 21 cột)
Sản phẩm được hiệu chỉnh xuất xứ: 493
Tỉ lệ Ngoài nước (sau hiệu chỉnh): 73.3%
Tỉ lệ Trong nước (sau hiệu chỉnh): 26.7%

--- Preview (5 dòng đầu) ---
                                                                  name_clean brand_name origin_class_corrected price_segment  rating  sold_count
2        Sữa rửa mặt dưỡng trắng cao cấp Hada Labo Premium Cleanser Radiance  Hada Labo             Ngoài nước   100k – 300k     5.0          23
3          Sữa rửa mặt Hada Labo dưỡng ẩm cao cấp Premium Cleanser Hydrating  Hada Labo             Ngoài nước     Dưới 100k     5.0          31
4                Sữa Rửa Mặt Rosette Làm Giảm Mụn Face Wash Pasta Acne Clear    Rosette             Ngoài nước   100k – 300k     5.0           7
5           Sữa rửa mặt X-Men Detox/Sáng da/Ngừa mụn/Kiểm soát nhờn/Mát lạnh      X-Men             Trong nước     Dư